In [1]:
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError
import sys

PROJECT_ROOT = Path.cwd()
DATA_PATH = (
    PROJECT_ROOT
    / "Cleaned- Master Data set"
    / "multilingual_ticket_master_cleaned.csv"
)

print("Python:", sys.version.split()[0])
print("Project folder:", PROJECT_ROOT)
print("Master dataset found:", DATA_PATH.is_file())

for package in ["pandas", "numpy", "scikit-learn", "boto3", "sagemaker"]:
    try:
        print(f"{package}: {version(package)}")
    except PackageNotFoundError:
        print(f"{package}: not installed")

Python: 3.12.14
Project folder: /home/sagemaker-user/AAI-540-Project/multilingual-it-ticket-routing-mlops
Master dataset found: True
pandas: 2.3.3
numpy: 1.26.4
scikit-learn: 1.7.2
boto3: 1.43.56
sagemaker: not installed


In [2]:
import pandas as pd

master_df = pd.read_csv(DATA_PATH, low_memory=False)

print("Dataset shape:", master_df.shape)

print("\nTickets per queue:")
print(master_df["queue"].value_counts())

print("\nTickets per language:")
print(master_df["language"].value_counts())

print("\nMissing queue labels:", master_df["queue"].isna().sum())

empty_text = (
    master_df["subject"].fillna("").str.strip().eq("")
    & master_df["body"].fillna("").str.strip().eq("")
)
print("Tickets with both subject and body empty:", empty_text.sum())

Dataset shape: (44278, 21)

Tickets per queue:
queue
Technical Support                  13095
Product Support                     8113
Customer Service                    6741
IT Support                          5237
Billing and Payments                4368
Returns and Exchanges               2204
Service Outages and Maintenance     1710
Sales and Pre-Sales                 1393
Human Resources                      802
General Inquiry                      615
Name: count, dtype: int64

Tickets per language:
language
en    25138
de    17379
es      812
fr      476
pt      473
Name: count, dtype: int64

Missing queue labels: 0
Tickets with both subject and body empty: 0


In [4]:
import boto3

aws_session = boto3.Session()
AWS_REGION = aws_session.region_name or "us-east-1"

sts_client = aws_session.client("sts", region_name=AWS_REGION)
identity = sts_client.get_caller_identity()

AWS_ACCOUNT_ID = identity["Account"]

print("Region:", AWS_REGION)
print("Current identity:", identity["Arn"])

Region: us-east-1
Current identity: arn:aws:sts::390568313988:assumed-role/LabRole/SageMaker


In [5]:
s3_client = aws_session.client("s3", region_name=AWS_REGION)

response = s3_client.list_buckets()
bucket_names = [bucket["Name"] for bucket in response["Buckets"]]

print("Existing S3 buckets:")
for name in bucket_names:
    print("-", name)

if not bucket_names:
    print("No buckets found.")

Existing S3 buckets:
- aws-athena-query-results-390568313988-us-east-1
- sagemaker-studio-h0k5mlt9jf
- sagemaker-us-east-1-390568313988


In [6]:
PROJECT_BUCKET = f"aai540-group3-tickets-{AWS_ACCOUNT_ID}-{AWS_REGION}"

# Check again so this cell can be rerun safely.
existing_buckets = {
    bucket["Name"]
    for bucket in s3_client.list_buckets()["Buckets"]
}

if PROJECT_BUCKET not in existing_buckets:
    create_options = {"Bucket": PROJECT_BUCKET}

    if AWS_REGION != "us-east-1":
        create_options["CreateBucketConfiguration"] = {
            "LocationConstraint": AWS_REGION
        }

    s3_client.create_bucket(**create_options)
    s3_client.get_waiter("bucket_exists").wait(Bucket=PROJECT_BUCKET)
    print("Created project bucket.")
else:
    print("Project bucket already exists.")

# Preserve versions of uploaded files.
s3_client.put_bucket_versioning(
    Bucket=PROJECT_BUCKET,
    VersioningConfiguration={"Status": "Enabled"}
)

versioning = s3_client.get_bucket_versioning(Bucket=PROJECT_BUCKET)

print("Bucket:", PROJECT_BUCKET)
print("Versioning:", versioning.get("Status"))

Created project bucket.
Bucket: aai540-group3-tickets-390568313988-us-east-1
Versioning: Enabled


In [7]:
import hashlib
import json
import subprocess

with DATA_PATH.open("rb") as file:
    dataset_sha256 = hashlib.file_digest(file, "sha256").hexdigest()

source_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    cwd=PROJECT_ROOT,
    text=True
).strip()

project_config = {
    "aws_region": AWS_REGION,
    "s3_bucket": PROJECT_BUCKET,
    "master_dataset": str(DATA_PATH.relative_to(PROJECT_ROOT)),
    "dataset_sha256": dataset_sha256,
    "source_commit": source_commit,
    "rows": len(master_df),
    "target": "queue",
    "split_fractions": {
        "train": 0.40,
        "validation": 0.10,
        "test": 0.10,
        "production": 0.40
    }
}

config_path = PROJECT_ROOT / "project_config.json"
config_path.write_text(json.dumps(project_config, indent=2), encoding="utf-8")

print("Configuration saved:", config_path.name)
print("Source commit:", source_commit[:12])
print("Dataset SHA-256:", dataset_sha256)

Configuration saved: project_config.json
Source commit: 59cbbdd7f3d2
Dataset fingerprint: c142eef90e9f58329ed6ac38f7a733350f3ca51bdb4aed4345a4baa8ea1a19aa


In [8]:
# Normalize whitespace and capitalization for duplicate checking.
ticket_text_check = (
    master_df["subject"].fillna("").astype(str)
    + " "
    + master_df["body"].fillna("").astype(str)
).str.replace(r"\s+", " ", regex=True).str.strip().str.casefold()

duplicate_check = pd.DataFrame({
    "text": ticket_text_check,
    "queue": master_df["queue"]
})

repeated_rows = duplicate_check["text"].duplicated().sum()

conflicting_groups = (
    duplicate_check.groupby("text")["queue"].nunique().gt(1).sum()
)

queue_language_counts = master_df.groupby(
    ["queue", "language"]
).size()

print("Repeated ticket rows beyond the first copy:", repeated_rows)
print("Text groups assigned to multiple queues:", conflicting_groups)

print("\nTicket counts by queue and language:")
print(queue_language_counts.unstack(fill_value=0))

Repeated ticket rows beyond the first copy: 0
Text groups assigned to multiple queues: 0

Ticket counts by queue and language:
language                           de    en   es   fr   pt
queue                                                     
Billing and Payments             1662  2540   83   37   46
Customer Service                 2693  3783  114   79   72
General Inquiry                   230   357    9    7   12
Human Resources                   303   476   11    7    5
IT Support                       2062  3000   81   48   46
Product Support                  3144  4664  136   96   73
Returns and Exchanges             871  1238   42   22   31
Sales and Pre-Sales               560   766   32   21   14
Service Outages and Maintenance   663   988   26   12   21
Technical Support                5191  7326  278  147  153


In [9]:
from sklearn.model_selection import train_test_split

split_df = master_df.copy()

# Give each ticket a stable identifier.
split_df["ticket_id"] = ticket_text_check.map(
    lambda text: hashlib.sha256(text.encode("utf-8")).hexdigest()
)

# Reserve 40% for simulated production.
development_df, production_df = train_test_split(
    split_df,
    test_size=0.40,
    stratify=split_df["queue"],
    random_state=42
)

# Training gets two-thirds of the remaining 60% = 40% overall.
train_df, evaluation_df = train_test_split(
    development_df,
    test_size=1 / 3,
    stratify=development_df["queue"],
    random_state=42
)

# Divide the remaining 20% into validation and test.
validation_df, test_df = train_test_split(
    evaluation_df,
    test_size=0.50,
    stratify=evaluation_df["queue"],
    random_state=42
)

splits = {
    "train": train_df,
    "validation": validation_df,
    "test": test_df,
    "production": production_df
}

for name, frame in splits.items():
    print(f"{name}: {len(frame):,} tickets ({len(frame) / len(split_df):.2%})")

# Confirm that every ticket appears in exactly one split.
all_ids = pd.concat([frame["ticket_id"] for frame in splits.values()])
assert len(all_ids) == len(master_df)
assert all_ids.is_unique

train: 17,710 tickets (40.00%)
validation: 4,428 tickets (10.00%)
test: 4,428 tickets (10.00%)
production: 17,712 tickets (40.00%)



In [10]:
SPLIT_DIR = PROJECT_ROOT / "data" / "splits"
LABEL_DIR = PROJECT_ROOT / "data" / "ground_truth"

SPLIT_DIR.mkdir(parents=True, exist_ok=True)
LABEL_DIR.mkdir(parents=True, exist_ok=True)

# Keep the fields needed for feature comparison.
input_columns = [
    "ticket_id", "subject", "body", "language", "priority", "type"
]

for name, frame in splits.items():
    columns = input_columns if name == "production" else input_columns + ["queue"]
    frame[columns].to_csv(SPLIT_DIR / f"{name}.csv", index=False)

production_df[["ticket_id", "queue"]].to_csv(
    LABEL_DIR / "production_labels.csv",
    index=False
)

# Store generated datasets in S3 instead of Git.
gitignore_path = PROJECT_ROOT / ".gitignore"
ignore_text = gitignore_path.read_text(encoding="utf-8")
if "/data/" not in ignore_text.splitlines():
    with gitignore_path.open("a", encoding="utf-8") as file:
        file.write("\n# Generated project datasets stored in S3\n/data/\n")

language_summary = pd.DataFrame({
    name: frame["language"].value_counts()
    for name, frame in splits.items()
}).fillna(0).astype(int)

print("Tickets by language and split:")
print(language_summary)
print("\nSaved four split files in:", SPLIT_DIR)
print("Saved production labels separately in:", LABEL_DIR)

Tickets by language and split:
          train  validation  test  production
language                                     
de         6989        1720  1720        6950
en         9994        2523  2555       10066
es          335          89    82         306
fr          200          42    39         195
pt          192          54    32         195

Saved four split files in: /home/sagemaker-user/AAI-540-Project/multilingual-it-ticket-routing-mlops/data/splits
Saved production labels separately in: /home/sagemaker-user/AAI-540-Project/multilingual-it-ticket-routing-mlops/data/ground_truth


In [11]:
S3_DATA_PREFIX = f"datasets/{dataset_sha256[:12]}/split-v1-seed42"

upload_files = {
    "master": (
        DATA_PATH,
        f"{S3_DATA_PREFIX}/cleaned/master.csv"
    ),
    **{
        name: (
            SPLIT_DIR / f"{name}.csv",
            f"{S3_DATA_PREFIX}/splits/{name}/{name}.csv"
        )
        for name in splits
    },
    "production_labels": (
        LABEL_DIR / "production_labels.csv",
        f"{S3_DATA_PREFIX}/ground_truth/production_labels.csv"
    )
}

s3_data = {}

for name, (local_path, s3_key) in upload_files.items():
    s3_client.upload_file(str(local_path), PROJECT_BUCKET, s3_key)

    uploaded = s3_client.head_object(
        Bucket=PROJECT_BUCKET,
        Key=s3_key
    )

    assert uploaded["ContentLength"] == local_path.stat().st_size
    assert uploaded.get("VersionId") not in (None, "null")

    s3_data[name] = {
        "uri": f"s3://{PROJECT_BUCKET}/{s3_key}",
        "version_id": uploaded["VersionId"]
    }

    print(f"Uploaded and checked: {name}")

project_config["s3_data"] = s3_data
project_config["split_seed"] = 42

config_path.write_text(
    json.dumps(project_config, indent=2),
    encoding="utf-8"
)

print("\nS3 file locations and version IDs saved in project_config.json")

Uploaded and checked: master
Uploaded and checked: train
Uploaded and checked: validation
Uploaded and checked: test
Uploaded and checked: production
Uploaded and checked: production_labels

S3 file locations and version IDs saved in project_config.json


In [12]:
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError
import json
import boto3
import pandas as pd

PROJECT_ROOT = Path.cwd()
config_path = PROJECT_ROOT / "project_config.json"
project_config = json.loads(config_path.read_text(encoding="utf-8"))

AWS_REGION = project_config["aws_region"]
PROJECT_BUCKET = project_config["s3_bucket"]

aws_session = boto3.Session(region_name=AWS_REGION)
s3_client = aws_session.client("s3")
athena_client = aws_session.client("athena")

print("Region:", AWS_REGION)
print("Project bucket:", PROJECT_BUCKET)

try:
    print("PyArrow:", version("pyarrow"))
except PackageNotFoundError:
    print("PyArrow: not installed")

Region: us-east-1
Project bucket: aai540-group3-tickets-390568313988-us-east-1
PyArrow: 21.0.0


In [13]:
from io import BytesIO
from urllib.parse import urlparse
import hashlib

# Read the recorded version of the master dataset from S3.
master_info = project_config["s3_data"]["master"]
master_uri = urlparse(master_info["uri"])

response = s3_client.get_object(
    Bucket=master_uri.netloc,
    Key=master_uri.path.lstrip("/"),
    VersionId=master_info["version_id"]
)

master_bytes = response["Body"].read()
response["Body"].close()

assert (
    hashlib.sha256(master_bytes).hexdigest()
    == project_config["dataset_sha256"]
), "Dataset file has changed. SHA-256 does not match."

catalog_df = pd.read_csv(BytesIO(master_bytes), low_memory=False)

# Give text columns consistent types for Parquet.
for column in catalog_df.select_dtypes(include="object").columns:
    catalog_df[column] = catalog_df[column].astype("string")

PARQUET_DIR = PROJECT_ROOT / "data" / "catalog"
PARQUET_DIR.mkdir(parents=True, exist_ok=True)
parquet_path = PARQUET_DIR / "master.parquet"

catalog_df.to_parquet(
    parquet_path, engine="pyarrow", compression="snappy", index=False
)

dataset_id = project_config["dataset_sha256"][:12]
PARQUET_PREFIX = f"catalog/{dataset_id}/master/"
parquet_key = PARQUET_PREFIX + "master.parquet"

s3_client.upload_file(str(parquet_path), PROJECT_BUCKET, parquet_key)

parquet_metadata = s3_client.head_object(
    Bucket=PROJECT_BUCKET, Key=parquet_key
)

assert parquet_metadata["ContentLength"] == parquet_path.stat().st_size

project_config["s3_data"]["master_parquet"] = {
    "uri": f"s3://{PROJECT_BUCKET}/{parquet_key}",
    "version_id": parquet_metadata["VersionId"]
}

config_path.write_text(
    json.dumps(project_config, indent=2), encoding="utf-8"
)

print("Rows and columns:", catalog_df.shape)
print("Parquet uploaded:", project_config["s3_data"]["master_parquet"]["uri"])

Rows and columns: (44278, 21)
Parquet uploaded: s3://aai540-group3-tickets-390568313988-us-east-1/catalog/c142eef90e9f/master/master.parquet


In [14]:
import time

ATHENA_DATABASE = "aai540_group3"
ATHENA_OUTPUT = f"s3://{PROJECT_BUCKET}/athena-results/"

def run_athena(sql, database=None):
    """Submit a SQL statement and wait up to 60 seconds."""
    options = {
        "QueryString": sql,
        "ResultConfiguration": {"OutputLocation": ATHENA_OUTPUT},
        "WorkGroup": "primary"
    }

    if database:
        options["QueryExecutionContext"] = {
            "Database": database,
            "Catalog": "AwsDataCatalog"
        }

    query_id = athena_client.start_query_execution(
        **options
    )["QueryExecutionId"]

    print("Query ID:", query_id)

    for _ in range(30):
        execution = athena_client.get_query_execution(
            QueryExecutionId=query_id
        )["QueryExecution"]

        status = execution["Status"]
        state = status["State"]

        if state == "SUCCEEDED":
            print("Status: SUCCEEDED")
            return query_id

        if state in ("FAILED", "CANCELLED"):
            raise RuntimeError(status.get("StateChangeReason", state))

        time.sleep(2)

    raise TimeoutError(
        f"Query is still pending. Check this ID before rerunning: {query_id}"
    )

database_query_id = run_athena(
    f"CREATE DATABASE IF NOT EXISTS {ATHENA_DATABASE}"
)

print("Database ready:", ATHENA_DATABASE)

Query ID: 4b37dcc1-e7da-4938-a30f-3d7772f1074c
Status: SUCCEEDED
Database ready: aai540_group3


In [15]:
import pyarrow as pa
import pyarrow.parquet as pq

ATHENA_TABLE = "tickets_master"
PARQUET_LOCATION = f"s3://{PROJECT_BUCKET}/{PARQUET_PREFIX}"

parquet_schema = pq.read_schema(parquet_path)

def athena_column_type(arrow_type):
    if pa.types.is_string(arrow_type) or pa.types.is_large_string(arrow_type):
        return "string"
    if pa.types.is_boolean(arrow_type):
        return "boolean"
    if pa.types.is_integer(arrow_type):
        return "bigint"
    if pa.types.is_floating(arrow_type):
        return "double"
    raise ValueError(f"Unsupported column type: {arrow_type}")

column_definitions = ",\n    ".join(
    f"`{field.name}` {athena_column_type(field.type)}"
    for field in parquet_schema
)

create_table_sql = f"""
CREATE EXTERNAL TABLE IF NOT EXISTS {ATHENA_DATABASE}.{ATHENA_TABLE} (
    {column_definitions}
)
STORED AS PARQUET
LOCATION '{PARQUET_LOCATION}'
"""

table_query_id = run_athena(
    create_table_sql,
    database=ATHENA_DATABASE
)

project_config["athena"] = {
    "database": ATHENA_DATABASE,
    "master_table": ATHENA_TABLE,
    "master_location": PARQUET_LOCATION,
    "query_output": ATHENA_OUTPUT,
    "workgroup": "primary"
}

config_path.write_text(
    json.dumps(project_config, indent=2),
    encoding="utf-8"
)

print("Table ready:", f"{ATHENA_DATABASE}.{ATHENA_TABLE}")
print("Registered columns:", len(parquet_schema))

Query ID: d65658b6-9044-4bba-aa78-581122d16a45
Status: SUCCEEDED
Table ready: aai540_group3.tickets_master
Registered columns: 21


In [16]:
summary_sql = f"""
SELECT
    COUNT(*) AS total_tickets,
    COUNT(DISTINCT queue) AS total_queues,
    COUNT(DISTINCT language) AS total_languages
FROM {ATHENA_DATABASE}.{ATHENA_TABLE}
"""

summary_query_id = run_athena(
    summary_sql,
    database=ATHENA_DATABASE
)

result = athena_client.get_query_results(
    QueryExecutionId=summary_query_id
)

rows = result["ResultSet"]["Rows"]
headers = [item["VarCharValue"] for item in rows[0]["Data"]]
values = [int(item["VarCharValue"]) for item in rows[1]["Data"]]

summary_df = pd.DataFrame([values], columns=headers)
display(summary_df)

assert values == [project_config["rows"], 10, 5], (
    "Athena counts do not match the expected dataset."
)


Query ID: a356b184-f37e-4318-8ff0-93ec35c33395
Status: SUCCEEDED


,total_tickets,total_queues,total_languages
0,44278,10,5


In [1]:
import json
from pathlib import Path
from datetime import datetime, timezone

import boto3
from botocore.exceptions import ClientError

PROJECT_ROOT = Path(
    "/home/sagemaker-user/AAI-540-Project/"
    "multilingual-it-ticket-routing-mlops"
)

config = json.loads(
    (PROJECT_ROOT / "project_config.json").read_text()
)

s3 = boto3.client("s3", region_name=config["aws_region"])
bucket = config["s3_bucket"]

security_checks = {
    "checked_at": datetime.now(timezone.utc).isoformat(),
    "bucket": bucket,
    "checks": {},
}

operations = {
    "versioning": s3.get_bucket_versioning,
    "default_encryption": s3.get_bucket_encryption,
    "bucket_public_access_block": s3.get_public_access_block,
}

for name, operation in operations.items():
    try:
        response = operation(Bucket=bucket)
        response.pop("ResponseMetadata", None)

        security_checks["checks"][name] = {
            "read_status": "Retrieved",
            "settings": response,
        }
    except ClientError as error:
        security_checks["checks"][name] = {
            "read_status": "Not verified",
            "error_code": error.response["Error"]["Code"],
            "message": error.response["Error"]["Message"],
        }

print(json.dumps(security_checks, indent=2))

{
  "checked_at": "2026-10-05T05:22:51.889267+00:00",
  "bucket": "aai540-group3-tickets-390568313988-us-east-1",
  "checks": {
    "versioning": {
      "read_status": "Retrieved",
      "settings": {
        "Status": "Enabled"
      }
    },
    "default_encryption": {
      "read_status": "Retrieved",
      "settings": {
        "ServerSideEncryptionConfiguration": {
          "Rules": [
            {
              "ApplyServerSideEncryptionByDefault": {
                "SSEAlgorithm": "AES256"
              },
              "BucketKeyEnabled": false,
              "BlockedEncryptionTypes": {
                "EncryptionType": [
                  "SSE-C"
                ]
              }
            }
          ]
        }
      }
    },
    "bucket_public_access_block": {
      "read_status": "Retrieved",
      "settings": {
        "PublicAccessBlockConfiguration": {
          "BlockPublicAcls": true,
          "IgnorePublicAcls": true,
          "BlockPublicPolicy": true,
       

In [2]:
security_report_dir = PROJECT_ROOT / "reports" / "security"
security_report_dir.mkdir(parents=True, exist_ok=True)

security_report_path = security_report_dir / "s3_security_checks.json"

security_report_path.write_text(
    json.dumps(security_checks, indent=2) + "\n",
    encoding="utf-8",
)

print("Security evidence saved:", security_report_path)

Security evidence saved: /home/sagemaker-user/AAI-540-Project/multilingual-it-ticket-routing-mlops/reports/security/s3_security_checks.json
